# ML Project : Predicting Hospital Readmission in Diabetic Patients
## Notebook 02 — Preprocessing, Modeling & Evaluation

**Dataset :** Diabetes 130-US Hospitals (1999–2008) — UCI ML Repository  
**Problem :** Binary Classification — will a diabetic patient be readmitted within 30 days?  

---

| Step | What we do |
|---|---|
| 1 | Import libraries |
| 2 | Load data |
| 3 | Clean data (duplicates, missing values, filter rows) |
| 4 | Feature engineering (5 new variables) |
| 5 | Build sklearn **Pipeline** (encode → scale → model) |
| 6 | Train 4 models and compare |
| 7 | Cross-validation |
| 8 | Hyperparameter optimization |
| 9 | Final evaluation + visualizations |
| 10 | Save the best model |

---
## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

# Sklearn — pipeline & preprocessing
from sklearn.pipeline          import Pipeline
from sklearn.preprocessing     import StandardScaler, LabelEncoder
from sklearn.impute             import SimpleImputer
from sklearn.compose            import ColumnTransformer

# Sklearn — model selection
from sklearn.model_selection   import train_test_split, StratifiedKFold, \
                                      cross_val_score, RandomizedSearchCV

# Sklearn — models
from sklearn.linear_model      import LogisticRegression
from sklearn.tree               import DecisionTreeClassifier
from sklearn.ensemble           import RandomForestClassifier, GradientBoostingClassifier

# Sklearn — metrics
from sklearn.metrics            import (accuracy_score, f1_score, precision_score,
                                         recall_score, roc_auc_score,
                                         classification_report, confusion_matrix,
                                         ConfusionMatrixDisplay, roc_curve,
                                         precision_recall_curve)

# Imbalanced learn — SMOTE
from imblearn.pipeline          import Pipeline as ImbPipeline
from imblearn.over_sampling     import SMOTE

import joblib
warnings.filterwarnings('ignore')

# Colors used in all charts
BLUE   = '#2E86AB'
RED    = '#E84855'
GREEN  = '#2ECC71'
ORANGE = '#F39C12'

print('Libraries loaded successfully')

ModuleNotFoundError: No module named 'pandas'

---
## 2. Load Data

In [ ]:
DATA_PATH = Path('..') / 'data' / 'diabetic_data.csv'

# na_values='?' : missing values are coded as '?' in this dataset
df = pd.read_csv(DATA_PATH, na_values='?')

print(f'Shape  : {df.shape[0]:,} rows  x  {df.shape[1]} columns')
print(f'Target distribution:')
print(df['readmitted'].value_counts().to_string())
df.head(3)

---
## 3. Clean Data

### 3.1 Remove Duplicates

In [ ]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)

print(f'Rows before : {before:,}')
print(f'Rows removed: {before - len(df):,}')
print(f'Rows after  : {len(df):,}')

### 3.2 Drop Useless Columns

| Column | Reason to drop |
|---|---|
| `encounter_id`, `patient_nbr` | Identifiers — no predictive value |
| `weight` | 97% missing |
| `payer_code` | 52% missing + administrative |
| `medical_specialty` | 53% missing + administrative |

In [ ]:
cols_to_drop = ['encounter_id', 'patient_nbr', 'weight',
                'payer_code', 'medical_specialty']

df.drop(columns=cols_to_drop, inplace=True)
print(f'Dropped {len(cols_to_drop)} columns. Remaining: {df.shape[1]}')

### 3.3 Filter Rows

Remove deceased / hospice patients — they **cannot** be readmitted, so including them would be clinically wrong.

In [ ]:
before = len(df)

# IDs 11,13,14,19,20,21 = deceased or hospice discharge
df = df[~df['discharge_disposition_id'].isin([11, 13, 14, 19, 20, 21])]
df = df[df['gender'].isin(['Male', 'Female'])]

print(f'Removed {before - len(df):,} rows (deceased, hospice, invalid gender)')
print(f'Remaining: {len(df):,} rows')

### 3.4 Create Binary Target

Original `readmitted` has 3 classes : `<30`, `>30`, `NO`  
We convert to binary : **1** = readmitted within 30 days | **0** = everything else

In [ ]:
df['target'] = (df['readmitted'] == '<30').astype(int)
df.drop(columns=['readmitted'], inplace=True)

print(f'Class 1 (readmitted <30d) : {df["target"].sum():,}  ({df["target"].mean()*100:.1f}%)')
print(f'Class 0 (others)          : {(df["target"]==0).sum():,}  ({(df["target"]==0).mean()*100:.1f}%)')
print('\n=> Severe class imbalance — handled by SMOTE inside the Pipeline')

---
## 4. Feature Engineering

We create **5 new variables** that capture clinical knowledge not directly visible in the raw data.

In [ ]:
# ── 1. ICD-9 Diagnosis Grouping ───────────────────────────────────────────
# 800+ raw codes → 8 clinical categories  (less noise, more signal)
def map_icd9(code):
    try:
        c = str(code).strip().upper()
        if c.startswith('V') or c.startswith('E'): return 'Other'
        n = float(c[:3])
        if   390 <= n <= 459: return 'Circulatory'
        elif 460 <= n <= 519: return 'Respiratory'
        elif 520 <= n <= 579: return 'Digestive'
        elif 250 <= n <= 250.99: return 'Diabetes'
        elif 800 <= n <= 999: return 'Injury'
        elif 710 <= n <= 739: return 'Musculoskeletal'
        elif 580 <= n <= 629: return 'Genitourinary'
        elif 140 <= n <= 239: return 'Neoplasms'
        else: return 'Other'
    except: return 'Other'

for col in ['diag_1', 'diag_2', 'diag_3']:
    df[col] = df[col].fillna('0').apply(map_icd9)

# ── 2. Age midpoint ───────────────────────────────────────────────────────
# '[60-70)' → 65   (numeric so linear models can use it)
age_map = {'[0-10)':5,'[10-20)':15,'[20-30)':25,'[30-40)':35,'[40-50)':45,
           '[50-60)':55,'[60-70)':65,'[70-80)':75,'[80-90)':85,'[90-100)':95}
df['age_mid'] = df['age'].map(age_map).fillna(55)
df.drop(columns=['age'], inplace=True)

# ── 3 & 4. Medication activity ───────────────────────────────────────────
# n_active_meds : how many drugs are currently prescribed
# n_med_changes : how many dosage changes happened (signals instability)
MED_COLS = ['metformin','repaglinide','nateglinide','chlorpropamide',
            'glimepiride','acetohexamide','glipizide','glyburide',
            'tolbutamide','pioglitazone','rosiglitazone','acarbose',
            'miglitol','troglitazone','tolazamide','examide',
            'sitagliptin','insulin','glyburide-metformin',
            'glipizide-metformin','glimepiride-pioglitazone',
            'metformin-rosiglitazone','metformin-pioglitazone']
MED_COLS = [c for c in MED_COLS if c in df.columns]

df['n_active_meds'] = (df[MED_COLS] != 'No').sum(axis=1)
df['n_med_changes']  = df[MED_COLS].isin(['Up','Down']).sum(axis=1)

# ── 5. Prior hospital visits total ───────────────────────────────────────
# outpatient + emergency + inpatient  (combined utilisation risk score)
df['prior_visits'] = (df['number_outpatient'] +
                      df['number_emergency']  +
                      df['number_inpatient'])

print('Feature engineering done. New features created:')
print('  diag_1/2/3  → ICD-9 clinical category')
print('  age_mid     → numeric age midpoint')
print('  n_active_meds, n_med_changes  → medication activity scores')
print('  prior_visits  → total prior hospital utilisation')
print(f'\nDataset shape after engineering: {df.shape}')

---
## 5. Prepare Features & Split

We identify which columns are **numeric** and which are **categorical** — the Pipeline will handle each differently.

In [ ]:
# Ordinal encode medication columns before split
# No=0, Steady=1, Up=2, Down=-1
med_order = {'No': 0, 'Steady': 1, 'Up': 2, 'Down': -1}
for col in MED_COLS:
    df[col] = df[col].map(med_order).fillna(0).astype(int)

# Label encode binary columns
for col in ['gender', 'change', 'diabetesMed']:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

# Fill remaining missing values
df['race'].fillna('Unknown', inplace=True)

# Identify numeric and categorical columns
X = df.drop(columns=['target'])
y = df['target']

NUM_COLS = X.select_dtypes(include='number').columns.tolist()
CAT_COLS = X.select_dtypes(include='object').columns.tolist()

print(f'Numeric features    : {len(NUM_COLS)}')
print(f'Categorical features: {len(CAT_COLS)}')
print(f'Total features      : {X.shape[1]}')

# Split: 80% train | 20% test — stratified to preserve 89/11 ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'\nTrain : {len(X_train):,} rows')
print(f'Test  : {len(X_test):,} rows')

---
## 6. Build the sklearn Pipeline

A **Pipeline** chains all preprocessing + model steps into one object.  
This guarantees:
- **No data leakage** — scaler and imputer only learn from training data
- **Clean code** — one `.fit()` call does everything
- **Easy deployment** — save one object, load one object

```
Pipeline structure:
  Step 1 — ColumnTransformer
    ├── Numeric columns   → SimpleImputer(median) → StandardScaler
    └── Categorical cols  → SimpleImputer(mode)   → OneHotEncoder
  Step 2 — SMOTE  (balance classes, train only)
  Step 3 — Classifier
```

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# ── Preprocessing sub-pipeline for numeric columns ────────────────────────
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),   # fill NaN with median
    ('scaler',  StandardScaler())                    # mean=0, std=1
])

# ── Preprocessing sub-pipeline for categorical columns ───────────────────
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),  # fill NaN with mode
    ('onehot',  OneHotEncoder(handle_unknown='ignore',     # one-hot encode
                               sparse_output=False))
])

# ── Combine both into a ColumnTransformer ─────────────────────────────────
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer,     NUM_COLS),
    ('cat', categorical_transformer, CAT_COLS)
])

print('Preprocessor ready:')
print(f'  Numeric  ({len(NUM_COLS)} cols) → Imputer(median) + StandardScaler')
print(f'  Categorical ({len(CAT_COLS)} cols) → Imputer(mode) + OneHotEncoder')

In [ ]:
# ── Define 4 full pipelines (preprocessor + SMOTE + model) ───────────────
# We use ImbPipeline (from imbalanced-learn) to include SMOTE inside the pipeline

pipelines = {

    'Logistic Regression': ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('smote',        SMOTE(random_state=42)),
        ('model',        LogisticRegression(
                             C=1.0,
                             max_iter=1000,
                             class_weight='balanced',
                             random_state=42))
    ]),

    'Decision Tree': ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('smote',        SMOTE(random_state=42)),
        ('model',        DecisionTreeClassifier(
                             max_depth=8,
                             min_samples_leaf=50,
                             class_weight='balanced',
                             random_state=42))
    ]),

    'Random Forest': ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('smote',        SMOTE(random_state=42)),
        ('model',        RandomForestClassifier(
                             n_estimators=200,
                             max_depth=10,
                             class_weight='balanced',
                             random_state=42,
                             n_jobs=-1))
    ]),

    'Gradient Boosting': ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('smote',        SMOTE(random_state=42)),
        ('model',        GradientBoostingClassifier(
                             n_estimators=150,
                             max_depth=4,
                             learning_rate=0.1,
                             random_state=42))
    ]),
}

print('4 Pipelines defined:')
for name in pipelines:
    print(f'  {name}')
print('\nEach pipeline = Preprocessor → SMOTE → Model')

---
## 7. Train All Models & Compare

In [ ]:
results = {}

print('Training all pipelines on the training set...\n')
print(f'{"Model":<22} {"Accuracy":>9} {"Precision":>10} {"Recall":>8} {"F1":>8} {"ROC-AUC":>9}')
print('-' * 72)

for name, pipe in pipelines.items():

    # One single .fit() call — handles imputation, scaling, SMOTE and training
    pipe.fit(X_train, y_train)

    # Predict on the TEST set
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]

    # Store results
    results[name] = {
        'pipe':      pipe,
        'y_pred':    y_pred,
        'y_prob':    y_prob,
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall':    recall_score(y_test, y_pred),
        'f1':        f1_score(y_test, y_pred),
        'roc_auc':   roc_auc_score(y_test, y_prob),
    }

    print(f"{name:<22} "
          f"{results[name]['accuracy']:>9.4f} "
          f"{results[name]['precision']:>10.4f} "
          f"{results[name]['recall']:>8.4f} "
          f"{results[name]['f1']:>8.4f} "
          f"{results[name]['roc_auc']:>9.4f}")

best_name = max(results, key=lambda n: results[n]['roc_auc'])
print(f'\n★ Best model by ROC-AUC: {best_name}  ({results[best_name]["roc_auc"]:.4f})')

In [ ]:
# Detailed classification report for each model
for name, res in results.items():
    print(f'\n{"="*55}')
    print(f'  {name}')
    print(f'{"="*55}')
    print(classification_report(
        y_test, res['y_pred'],
        target_names=['Not readmitted <30d', 'Readmitted <30d']
    ))

In [ ]:
# Confusion matrices for all 4 models
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
cmaps = ['Blues', 'Greens', 'Oranges', 'Reds']
fig.suptitle('Confusion Matrices — All Models (Test Set)',
             fontsize=13, fontweight='bold')

for ax, (name, res), cmap in zip(axes, results.items(), cmaps):
    cm = confusion_matrix(y_test, res['y_pred'])
    ConfusionMatrixDisplay(
        cm, display_labels=['Not <30d', 'Readmitted <30d']
    ).plot(ax=ax, colorbar=False, cmap=cmap)
    ax.set_title(name, fontsize=9, fontweight='bold')
    ax.set_xticklabels(['Not <30d', '<30d'], rotation=0)
    ax.set_yticklabels(['Not <30d', '<30d'], rotation=0)

plt.tight_layout()
plt.show()

---
## 8. Cross-Validation

A single train/test split can be lucky or unlucky.  
**5-fold cross-validation** repeats the evaluation 5 times and gives a more reliable estimate.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('5-Fold Cross-Validation (scoring = ROC-AUC)\n')
print(f'  {"Model":<22} {"Mean AUC":>10} {"Std AUC":>10}')
print('  ' + '-' * 45)

for name, pipe in pipelines.items():
    scores = cross_val_score(
        pipe, X_train, y_train,
        cv=cv, scoring='roc_auc', n_jobs=-1
    )
    print(f'  {name:<22}  {scores.mean():.4f}  ±  {scores.std():.4f}')

---
## 9. Hyperparameter Optimization (Best Model)

**RandomizedSearchCV** automatically explores combinations of hyperparameters  
and returns the best configuration based on cross-validation.

In [ ]:
from scipy.stats import randint, uniform

print(f'Optimizing: {best_name}\n')

# Parameter grids — prefix 'model__' targets the last step of the pipeline
param_grids = {
    'Logistic Regression': {
        'model__C': uniform(0.01, 10)
    },
    'Decision Tree': {
        'model__max_depth':        randint(4, 15),
        'model__min_samples_leaf': randint(20, 100),
    },
    'Random Forest': {
        'model__n_estimators':     randint(100, 300),
        'model__max_depth':        randint(8, 20),
        'model__min_samples_leaf': randint(15, 50),
    },
    'Gradient Boosting': {
        'model__n_estimators':  randint(100, 250),
        'model__max_depth':     randint(3, 7),
        'model__learning_rate': uniform(0.05, 0.15),
    }
}

search = RandomizedSearchCV(
    pipelines[best_name],         # the full pipeline
    param_distributions=param_grids[best_name],
    n_iter=20,                    # test 20 random combinations
    scoring='roc_auc',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search.fit(X_train, y_train)

print(f'\nBest hyperparameters:')
for k, v in search.best_params_.items():
    print(f'  {k}: {v}')
print(f'\nBest CV ROC-AUC: {search.best_score_:.4f}')

best_pipeline = search.best_estimator_

---
## 10. Final Evaluation (Best Optimized Model)

In [ ]:
# Predict on the test set using the optimized pipeline
y_pred_best = best_pipeline.predict(X_test)
y_prob_best = best_pipeline.predict_proba(X_test)[:, 1]

print(f'Final Results — {best_name} (optimized) — Test Set')
print(f'  Accuracy  : {accuracy_score(y_test, y_pred_best):.4f}')
print(f'  Precision : {precision_score(y_test, y_pred_best, zero_division=0):.4f}')
print(f'  Recall    : {recall_score(y_test, y_pred_best):.4f}')
print(f'  F1-Score  : {f1_score(y_test, y_pred_best):.4f}')
print(f'  ROC-AUC   : {roc_auc_score(y_test, y_prob_best):.4f}')
print()
print(classification_report(
    y_test, y_pred_best,
    target_names=['Not readmitted <30d', 'Readmitted <30d']
))

In [ ]:
# ── Visualizations ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Final Evaluation — {best_name}', fontsize=13, fontweight='bold')

# Fig 1 : Model comparison bar chart
names = list(results.keys())
f1s   = [results[n]['f1']      for n in names]
aucs  = [results[n]['roc_auc'] for n in names]
x     = np.arange(len(names))
axes[0].bar(x - 0.2, f1s,  0.35, label='F1',      color=BLUE)
axes[0].bar(x + 0.2, aucs, 0.35, label='ROC-AUC', color=RED)
axes[0].set_xticks(x)
axes[0].set_xticklabels(names, rotation=20, ha='right', fontsize=9)
axes[0].set_ylim(0, 1.1)
axes[0].set_title('All Models Comparison')
axes[0].legend()

# Fig 2 : Confusion matrix of best model
cm = confusion_matrix(y_test, y_pred_best)
ConfusionMatrixDisplay(
    cm, display_labels=['Not <30d', 'Readmitted <30d']
).plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title(f'Confusion Matrix\n{best_name}')

# Fig 3 : ROC curves all models
for (name, res), color in zip(results.items(), [BLUE, ORANGE, GREEN, RED]):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    lw = 2.5 if name == best_name else 1.2
    axes[2].plot(fpr, tpr, lw=lw, color=color,
                 label=f'{name}  (AUC={res["roc_auc"]:.3f})')
axes[2].plot([0,1],[0,1],'k--',lw=1)
axes[2].set_xlabel('False Positive Rate')
axes[2].set_ylabel('True Positive Rate')
axes[2].set_title('ROC Curves')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance (if the best model supports it)
model_step = best_pipeline.named_steps['model']

if hasattr(model_step, 'feature_importances_'):
    # Get feature names after preprocessing
    ohe_features = (best_pipeline.named_steps['preprocessor']
                    .named_transformers_['cat']
                    .named_steps['onehot']
                    .get_feature_names_out(CAT_COLS).tolist())
    all_feature_names = NUM_COLS + ohe_features

    fi = pd.Series(
        model_step.feature_importances_,
        index=all_feature_names
    ).sort_values(ascending=False).head(15)

    fig, ax = plt.subplots(figsize=(10, 6))
    fi[::-1].plot(kind='barh', ax=ax, color=BLUE, edgecolor='white')
    ax.set_title(f'Top 15 Feature Importances — {best_name}',
                 fontweight='bold')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.show()

    print('Top 10 most important features:')
    for i, (feat, imp) in enumerate(fi.head(10).items(), 1):
        print(f'  {i:2d}. {feat:<45s} {imp:.5f}')

elif hasattr(model_step, 'coef_'):
    ohe_features = (best_pipeline.named_steps['preprocessor']
                    .named_transformers_['cat']
                    .named_steps['onehot']
                    .get_feature_names_out(CAT_COLS).tolist())
    all_feature_names = NUM_COLS + ohe_features

    fi = pd.Series(
        np.abs(model_step.coef_[0]),
        index=all_feature_names
    ).sort_values(ascending=False).head(15)

    fig, ax = plt.subplots(figsize=(10, 6))
    fi[::-1].plot(kind='barh', ax=ax, color=BLUE, edgecolor='white')
    ax.set_title(f'Top 15 Features (|Coefficient|) — {best_name}',
                 fontweight='bold')
    ax.set_xlabel('|Coefficient|')
    plt.tight_layout()
    plt.show()

---
## 11. Save the Best Model

In [ ]:
MODEL_DIR = Path('..') / 'models'
MODEL_DIR.mkdir(exist_ok=True)

# Save the complete pipeline (preprocessor + SMOTE + model — all in one)
joblib.dump(best_pipeline, MODEL_DIR / 'best_pipeline.pkl')

print('Pipeline saved successfully.')
print(f'  Location : {(MODEL_DIR / "best_pipeline.pkl").resolve()}')
print()
print('To use on a new patient:')
print("""
  pipeline = joblib.load('models/best_pipeline.pkl')
  prediction = pipeline.predict(new_patient_df)
  # 0 = low risk | 1 = HIGH RISK → schedule follow-up
""")

---
## 12. Summary

In [ ]:
print('=' * 60)
print('PROJECT SUMMARY')
print('=' * 60)
print(f'Dataset         : {len(df):,} patient records')
print(f'Features used   : {X.shape[1]}')
print(f'Engineered feat : 5 new clinical variables')
print(f'Algorithms      : Logistic Regression, Decision Tree,')
print(f'                  Random Forest, Gradient Boosting')
print(f'Optimization    : RandomizedSearchCV (20 iter x 3 folds)')
print()
print(f'BEST MODEL : {best_name}')
print(f'  ROC-AUC  : {roc_auc_score(y_test, y_prob_best):.4f}')
print(f'  F1-Score : {f1_score(y_test, y_pred_best):.4f}')
print(f'  Recall   : {recall_score(y_test, y_pred_best):.4f}')
print('=' * 60)